# Predicting Denver Traffic Accident Severity

In [1]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, VotingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, roc_auc_score,
    classification_report,
)
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.metrics import average_precision_score


# Load data.
df = pd.read_csv("traffic_accident_data.csv", low_memory=False)

# Create the target variable before dropping leakage columns.
# Create high risk variable.

# Fill NaN as 0 so the comparison is safe
si = df["SERIOUSLY_INJURED"].fillna(0)
fat = df["FATALITIES"].fillna(0)

# Create a field where any fatality or serious injury
# is classified as "high risk".
df["high_risk"] = ((si > 0) | (fat > 0)).astype(int)

#df["top_traffic_accident_offense"].unique()

# Drop columns. I am taking a guess by removing these
# columns which appear superflous. The leakage columns
# indicate data we are trying to predict, so we omit them.

ID_COLS = [
    "object_id", "incident_id", "offense_id",
    "incident_address", "reported_date",
    "last_occurrence_date", "POINT_X", "POINT_Y", "x", "y",
    "offense_code", "offense_code_extension", 
    "geo_x", "geo_y", # redundant data.
]

LEAKAGE_COLS = [
    "SERIOUSLY_INJURED", "FATALITIES",
    "FATALITY_MODE_1", "FATALITY_MODE_2",
    "SERIOUSLY_INJURED_MODE_1", "SERIOUSLY_INJURED_MODE_2",
    "top_traffic_accident_offense",
    #"HARMFUL_EVENT_SEQ_1", "HARMFUL_EVENT_SEQ_2", "HARMFUL_EVENT_SEQ_3",
]

cols_to_drop = [c for c in ID_COLS + LEAKAGE_COLS]
df.drop(columns=cols_to_drop, inplace=True)

#print(df["first_occurrence_date"].unique())

# It is necessary to separate the time of the incident 
# into multiple categories.

df["first_occurrence_date"] = pd.to_datetime(
    df["first_occurrence_date"], errors="coerce"
)

df["hour"] = df["first_occurrence_date"].dt.hour
df["day_of_week"] = df["first_occurrence_date"].dt.dayofweek   # 0=Mon 6=Sun
df["month"] = df["first_occurrence_date"].dt.month

# We no longer need the raw datetime column
df.drop(columns=["first_occurrence_date"], inplace=True)

# Truly continuous — keep numeric, StandardScaler is appropriate
#true_num_cols = ['geo_lat', 'geo_lon']

# The nature of these features is unclear to me. To keep things simple
# we will make them boolean.
df['bicycle_ind']   = (df['bicycle_ind'] > 0).astype(int)
df['pedestrian_ind'] = (df['pedestrian_ind'] > 0).astype(int)

# These columns, while numbers, are not necessarily numeric. 
# They should be one-hot-encoded. I'll turn them into strings.
# so they get picked up by select_dtypes(include='object')
for col in ['precinct_id', 'hour', 'day_of_week', 'month']:
    df[col] = df[col].astype(str)

# Clean categorical columns

# selecting for object dtype gives us features with strings in them 
# These will be one hot encoded.
cat = df.select_dtypes(include="object").columns.tolist()

# Normalize strings
for col in cat:
    df[col] = (
        df[col]
        .astype(str)         # ensure string type
        .str.strip()         # remove leading/trailing whitespace
        .str.upper()         # normalize to uppercase
        .replace("NAN", np.nan)   # convert "nan" strings back to NaN
        .replace("", np.nan)      # blank strings -> NaN
    )

# Handle missing values

# Separate feature types (excluding the target)
feature_cols = [c for c in df.columns if c != "high_risk"]
cat_cols  = df[feature_cols].select_dtypes(include="object").columns.tolist()
num_cols  = df[feature_cols].select_dtypes(include=["number"]).columns.tolist()

# Fill categoricals with "UNKNOWN". By doing this, we can keep dataponits we would
# otherwise have to drop.
df[cat_cols] = df[cat_cols].fillna("UNKNOWN")

# Fill numeric columns with median. It's a safe choice.
for col in num_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)


# Separate features X and target y

X = df.drop(columns=["high_risk"])
y = df["high_risk"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y,      # keep class balance in both splits
)

# Re-identify column types from training data only
final_cat_cols = X_train.select_dtypes(include="object").columns.tolist()
final_num_cols = X_train.select_dtypes(include="number").columns.tolist()

# Build ColumnTransformer preprocessor

preprocessor = ColumnTransformer(
    transformers=[
        # One-hot encode categorical columns; ignore unseen categories
        ("cat", OneHotEncoder(handle_unknown="ignore"), final_cat_cols), # sparse_output=False
        ("num", StandardScaler(), final_num_cols),
    ],
    remainder="drop",   # Discard any column not listed above.
)

In [2]:
def evaluate_model(name, pipeline, X_train, y_train, X_test, y_test, thr=0.5):
    """
    Parameters:
        string: name of classifier,
        Pipeline: classifier pipeline,
        X_train,
        y_train,
        X_test,
        y_test,
        int: threshold value (0..1)

    Returns:
        fitted pipeline,
        dictionary struct of important metrics
        
    Fit pipeline on training data, then print full evaluation on test data.
    ASSUME ALL MODELS HAVE predict_proba() FUNCTION
    
    """

    pipeline.fit(X_train, y_train)
    #y_pred = pipeline.predict(X_test)

    # generate the predictions using each models'
    # probability prediction for the samples. This 
    # way we can alter the threshold to improve recall
    # and we can use these models for a soft voter.
    y_prob = pipeline.predict_proba(X_test)[:, 1]
    y_pred = (y_prob >= thr).astype(int)

    auc = roc_auc_score(y_test, y_prob)
    auc_str = f"{auc:.4f}"
    
    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec  = recall_score(y_test, y_pred, zero_division=0)
    f1   = f1_score(y_test, y_pred, zero_division=0)
    cm   = confusion_matrix(y_test, y_pred)
    prauc  = average_precision_score(y_test, y_prob)

    # Fancy printing done by Mr. Chat.
    print(f"\n{'─'*65}")
    print(f"  MODEL: {name}")
    print(f"{'─'*65}")
    print(f"  Accuracy  : {acc:.4f}")
    print(f"  Precision : {prec:.4f}   (of predicted high-risk, how many truly are)")
    print(f"  Recall    : {rec:.4f}   (of actual high-risk, how many we caught)")
    print(f"  F1-Score  : {f1:.4f}   (harmonic mean of precision & recall)")
    print(f"  ROC-AUC   : {auc_str}")
    print(f"  RC-AUC   : {prauc:.4f}")
    print()
    print("  Confusion matrix:")
    print(f"    {'':20s}  Pred=0   Pred=1")
    print(f"    Actual=0 (low risk)   {cm[0,0]:>6,}   {cm[0,1]:>6,}")
    print(f"    Actual=1 (high risk)  {cm[1,0]:>6,}   {cm[1,1]:>6,}")
    print()
    print("  Full classification report:")
    print(classification_report(y_test, y_pred, target_names=["Low risk", "High risk"]))

    return pipeline, {"name": name, "accuracy": acc, "precision": prec,
                      "recall": rec, "f1": f1, "auc": auc_str}

# Define Models
## What makes a good model for this problem?
This problem is essentially a classification problem. We are looking for situations that are considered either "low risk" or "high risk". Additionally, because the theoretical outcome of accidentally classifying a high risk accident as a low risk one is harmful, the important metrics to look at are recall, and PR-AUC, since the positive class is rare.

Firstly, I want to showcase models that didn't work too well

# Dummy Classifier
This classifier acts as a control model that shows us a naive approach to classification and the resulting metrics.

In [11]:
# Model 1, a dummy classifier that will always choose low risk.
# Acts as a control group.
dummy_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", DummyClassifier(strategy="most_frequent", random_state=42)),
])

dummy_pipe_fitted, dummy_metrics = evaluate_model(
    "DummyClassifier (control)", dummy_pipe, X_train, y_train, X_test, y_test
)


─────────────────────────────────────────────────────────────────
  MODEL: DummyClassifier (control)
─────────────────────────────────────────────────────────────────
  Accuracy  : 0.9767
  Precision : 0.0000   (of predicted high-risk, how many truly are)
  Recall    : 0.0000   (of actual high-risk, how many we caught)
  F1-Score  : 0.0000   (harmonic mean of precision & recall)
  ROC-AUC   : 0.5000
  RC-AUC   : 0.0233

  Confusion matrix:
                          Pred=0   Pred=1
    Actual=0 (low risk)   54,936        0
    Actual=1 (high risk)   1,308        0

  Full classification report:
              precision    recall  f1-score   support

    Low risk       0.98      1.00      0.99     54936
   High risk       0.00      0.00      0.00      1308

    accuracy                           0.98     56244
   macro avg       0.49      0.50      0.49     56244
weighted avg       0.95      0.98      0.97     56244



# Naive Bayes
Naive Bayes didn't strike me as a model that would perform well on our dataset and problem, and I was proving right. It's recall for "high risk" samples is amazing, but it comes at the cost of abysmal precision. This classifier almost has the opposite problem of the Dummy Classifier, in that it claims everything is high risk! Interestingly enough, it is also the only classifier that required me to alter my preprocessor, as it needed the oneHotEncoder to have a dense output.

In [7]:
from sklearn.naive_bayes import GaussianNB

# Naive Bayes assumes all features are independent of each other given the
# class label — a strong assumption that is almost certainly wrong here
# (e.g. vehicle type and driver action are related). Despite this, NB often
# performs surprisingly well and is extremely fast to train.
#
# GaussianNB assumes numeric features follow a Gaussian distribution.
# After OneHotEncoding, our categorical features become 0/1 columns which
# are not Gaussian — this is a known limitation. BernoulliNB would be more
# theoretically correct for binary features, but GaussianNB tends to work
# fine in practice on mixed encoded data.
#
# var_smoothing adds a small value to variances for numerical stability.

gnb_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", GaussianNB(
        var_smoothing=1e-9,    # default — increase if you get numerical errors
    )),
])

gnb_pipe_fitted, gnb_metrics = evaluate_model(
    "Gaussian Naive Bayes", gnb_pipe, X_train, y_train, X_test, y_test
)


─────────────────────────────────────────────────────────────────
  MODEL: Gaussian Naive Bayes
─────────────────────────────────────────────────────────────────
  Accuracy  : 0.0791
  Precision : 0.0243   (of predicted high-risk, how many truly are)
  Recall    : 0.9870   (of actual high-risk, how many we caught)
  F1-Score  : 0.0475   (harmonic mean of precision & recall)
  ROC-AUC   : 0.5249

  Confusion matrix:
                          Pred=0   Pred=1
    Actual=0 (low risk)    3,156   51,780
    Actual=1 (high risk)      17    1,291

  Full classification report:
              precision    recall  f1-score   support

    Low risk       0.99      0.06      0.11     54936
   High risk       0.02      0.99      0.05      1308

    accuracy                           0.08     56244
   macro avg       0.51      0.52      0.08     56244
weighted avg       0.97      0.08      0.11     56244



# MLP Classifier
This classifier uses artificial-neural networks to do its job. I thought it would have a chance, but unfortunately it also performed very poorly.

In [16]:
from sklearn.neural_network import MLPClassifier

# A multi-layer perceptron — a simple feedforward neural network.
# (100, 50) means two hidden layers: 100 neurons then 50 neurons.
# StandardScaler in the preprocessor is important here — MLPs are
# sensitive to feature scale, which we're already handling.
#
# MLPClassifier has no class_weight parameter, so we pass sample_weight
# through evaluate_model won't handle that easily — we instead use a
# larger network and rely on the scaled features. If recall is too low,
# consider SMOTE oversampling before fitting.

mlp_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", MLPClassifier(
        hidden_layer_sizes=(100, 50),  # two hidden layers
        activation="relu",
        max_iter=300,
        random_state=42,
        early_stopping=True,       # stops if validation loss stops improving
        validation_fraction=0.1,
        n_iter_no_change=15,
    )),
])

mlp_pipe_fitted, mlp_metrics = evaluate_model(
    "MLP Classifier", mlp_pipe, X_train, y_train, X_test, y_test
)


─────────────────────────────────────────────────────────────────
  MODEL: MLP Classifier
─────────────────────────────────────────────────────────────────
  Accuracy  : 0.9767
  Precision : 0.4941   (of predicted high-risk, how many truly are)
  Recall    : 0.1284   (of actual high-risk, how many we caught)
  F1-Score  : 0.2039   (harmonic mean of precision & recall)
  ROC-AUC   : 0.8877

  Confusion matrix:
                          Pred=0   Pred=1
    Actual=0 (low risk)   54,764      172
    Actual=1 (high risk)   1,140      168

  Full classification report:
              precision    recall  f1-score   support

    Low risk       0.98      1.00      0.99     54936
   High risk       0.49      0.13      0.20      1308

    accuracy                           0.98     56244
   macro avg       0.74      0.56      0.60     56244
weighted avg       0.97      0.98      0.97     56244



# The Good Models
Now that we've seen some models that are poorly suited for our purposes, let's look at some that showed more promise-and what we did to improve them.

# Logistic Regression
Initially, LR has proven itself to be one of our finest models yet. running a parameter Grid search found for us that C=0.01 is most optimal for our data.

In [3]:
# Model 2, logistic regression

lr_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000, 
        class_weight="balanced", 
        C=0.01, random_state=42, # C was given to us through grid search. (thanks Mark)
        #solver="saga",
        #penalty,
    )),
])

lr_pipe_fitted, lr_metrics = evaluate_model(
    "Logistic Regression", lr_pipe, X_train, y_train, X_test, y_test #, thr=0.3
)


─────────────────────────────────────────────────────────────────
  MODEL: Logistic Regression
─────────────────────────────────────────────────────────────────
  Accuracy  : 0.8302
  Precision : 0.0992   (of predicted high-risk, how many truly are)
  Recall    : 0.7798   (of actual high-risk, how many we caught)
  F1-Score  : 0.1760   (harmonic mean of precision & recall)
  ROC-AUC   : 0.8880
  RC-AUC   : 0.2306

  Confusion matrix:
                          Pred=0   Pred=1
    Actual=0 (low risk)   45,674    9,262
    Actual=1 (high risk)     288    1,020

  Full classification report:
              precision    recall  f1-score   support

    Low risk       0.99      0.83      0.91     54936
   High risk       0.10      0.78      0.18      1308

    accuracy                           0.83     56244
   macro avg       0.55      0.81      0.54     56244
weighted avg       0.97      0.83      0.89     56244



# Random Forest

In [4]:
# Model 3, Random Forest 
rf_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=500,
        class_weight="balanced_subsample",
        random_state=42,
        n_jobs=-1,
        #max_depth=20,
        min_samples_leaf=5,
    )),
])


rf_pipe_fitted, rf_metrics = evaluate_model(
    "Random Forest", rf_pipe, X_train, y_train, X_test, y_test
)



─────────────────────────────────────────────────────────────────
  MODEL: Random Forest
─────────────────────────────────────────────────────────────────
  Accuracy  : 0.9629
  Precision : 0.2747   (of predicted high-risk, how many truly are)
  Recall    : 0.3616   (of actual high-risk, how many we caught)
  F1-Score  : 0.3122   (harmonic mean of precision & recall)
  ROC-AUC   : 0.8881
  RC-AUC   : 0.2261

  Confusion matrix:
                          Pred=0   Pred=1
    Actual=0 (low risk)   53,687    1,249
    Actual=1 (high risk)     835      473

  Full classification report:
              precision    recall  f1-score   support

    Low risk       0.98      0.98      0.98     54936
   High risk       0.27      0.36      0.31      1308

    accuracy                           0.96     56244
   macro avg       0.63      0.67      0.65     56244
weighted avg       0.97      0.96      0.97     56244



# Extra Trees: A type of random Forest

In [5]:
# Extra Tree Classifier
et_pipe = Pipeline([
    ("pre", preprocessor),
    ("clf", ExtraTreesClassifier(
        n_estimators=500,
        class_weight="balanced_subsample",
        random_state=42,
        n_jobs=-1,
        #max_depth=20,
        min_samples_leaf=5,
    )),
])

"""
calibrated_et = CalibratedClassifierCV(
    estimator=et_pipe,
    method="isotonic",
    cv=3
)

et_pipe_fitted, et_metrics = evaluate_model(
    "Extra Trees", calibrated_et, X_train, y_train, X_test, y_test
)
"""

et_pipe_fitted, et_metrics = evaluate_model(
    "Extra Trees", et_pipe, X_train, y_train, X_test, y_test
)


─────────────────────────────────────────────────────────────────
  MODEL: Extra Trees
─────────────────────────────────────────────────────────────────
  Accuracy  : 0.9577
  Precision : 0.2547   (of predicted high-risk, how many truly are)
  Recall    : 0.4251   (of actual high-risk, how many we caught)
  F1-Score  : 0.3185   (harmonic mean of precision & recall)
  ROC-AUC   : 0.8905
  RC-AUC   : 0.2317

  Confusion matrix:
                          Pred=0   Pred=1
    Actual=0 (low risk)   53,309    1,627
    Actual=1 (high risk)     752      556

  Full classification report:
              precision    recall  f1-score   support

    Low risk       0.99      0.97      0.98     54936
   High risk       0.25      0.43      0.32      1308

    accuracy                           0.96     56244
   macro avg       0.62      0.70      0.65     56244
weighted avg       0.97      0.96      0.96     56244



# XGBoost


In [6]:
from xgboost import XGBClassifier

# XGBoost is a gradient boosting model — like AdaBoost but more sophisticated.
# It builds trees sequentially, but uses gradient descent to minimize a loss
# function rather than just reweighting misclassified samples.
#
# scale_pos_weight handles class imbalance: set it to the ratio of
# negative to positive samples. With ~97.7% low-risk and ~2.3% high-risk,
# that's roughly 97.7/2.3 ≈ 42.
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale = neg / pos  # ~42

xgb_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", XGBClassifier(
        n_estimators=200,
        max_depth=4,               # shallow trees work well with boosting
        learning_rate=0.1,
        scale_pos_weight=scale,    # handles class imbalance
        random_state=42,
        n_jobs=-1,
        eval_metric="auc",         # suppresses a default warning
        verbosity=0,
    )),
])

xgb_pipe_fitted, xgb_metrics = evaluate_model(
    "XGBoost", xgb_pipe, X_train, y_train, X_test, y_test
)


─────────────────────────────────────────────────────────────────
  MODEL: XGBoost
─────────────────────────────────────────────────────────────────
  Accuracy  : 0.8378
  Precision : 0.0995   (of predicted high-risk, how many truly are)
  Recall    : 0.7416   (of actual high-risk, how many we caught)
  F1-Score  : 0.1754   (harmonic mean of precision & recall)
  ROC-AUC   : 0.8821
  RC-AUC   : 0.2464

  Confusion matrix:
                          Pred=0   Pred=1
    Actual=0 (low risk)   46,154    8,782
    Actual=1 (high risk)     338      970

  Full classification report:
              precision    recall  f1-score   support

    Low risk       0.99      0.84      0.91     54936
   High risk       0.10      0.74      0.18      1308

    accuracy                           0.84     56244
   macro avg       0.55      0.79      0.54     56244
weighted avg       0.97      0.84      0.89     56244



In [7]:
import pickle
import os

os.makedirs("pickle_jar", exist_ok=True)

# Save each fitted pipeline
with open("pickle_jar/lr_pipe_fitted.pkl", "wb") as f:
    pickle.dump(lr_pipe_fitted, f)

with open("pickle_jar/rf_pipe_fitted.pkl", "wb") as f:
    pickle.dump(rf_pipe_fitted, f)
    
with open("pickle_jar/et_pipe_fitted.pkl", "wb") as f:
    pickle.dump(et_pipe_fitted, f)

with open("pickle_jar/xgb_pipe_fitted.pkl", "wb") as f:
    pickle.dump(xgb_pipe_fitted, f)

print("Models saved.")

Models saved.


# Combining these models
These models performed decently. However, is there a way we can increase the performance even more? The plan is to create an ensemble learner out of these models. Each of these models has a function called `predict_proba()`and what this does is it outputs the probability or confidence that the model has for a sample being of the positive class. What we can do is implement a soft voting, where instead of a hard vote, where the popular prediction is chosen, we can make our prediction based on the average of all of the predictions. This should improve our model even more.

In [18]:
# Each model is already fitted from your individual cells above.
# Just extract probabilities and average — no VotingClassifier needed.

lr_probs  = lr_pipe_fitted.predict_proba(X_test)[:, 1]
rf_probs  = rf_pipe_fitted.predict_proba(X_test)[:, 1]
et_probs  = et_pipe_fitted.predict_proba(X_test)[:, 1]
xgb_probs = xgb_pipe_fitted.predict_proba(X_test)[:, 1]


# Average — each model gets equal weight
# ensemble_probs = (lr_probs + rf_probs + et_probs + xgb_probs) / 4.0
ensemble_probs = (lr_probs + xgb_probs + rf_probs) / 3.0

# Sweep thresholds to find the best operating point
print(f"{'Threshold':<12} {'Precision':<12} {'Recall':<10} {'F1':<10} {'AUC':<10} {'TP':<8} {'FN'}")
print("─" * 70)
for thr in [0.5, 0.4, 0.3, 0.25, 0.2, 0.15]:
    preds = (ensemble_probs >= thr).astype(int)
    cm    = confusion_matrix(y_test, preds)
    tn, fp, fn, tp = cm.ravel()
    print(f"t={thr:<10} "
          f"{precision_score(y_test, preds, zero_division=0):<12.3f}"
          f"{recall_score(y_test, preds, zero_division=0):<10.3f}"
          f"{f1_score(y_test, preds, zero_division=0):<10.3f}"
          f"{roc_auc_score(y_test, ensemble_probs):<10.4f}"
          f"{tp:<8} {fn}")
    print(f"  Confusion matrix:")
    print(f"    {'':22s}  Pred=0   Pred=1")
    print(f"    Actual=0 (low risk)   {tn:>6,}   {fp:>6,}")
    print(f"    Actual=1 (high risk)  {fn:>6,}   {tp:>6,}")

Threshold    Precision    Recall     F1         AUC        TP       FN
──────────────────────────────────────────────────────────────────────
t=0.5        0.141       0.670     0.233     0.8913    876      432
  Confusion matrix:
                            Pred=0   Pred=1
    Actual=0 (low risk)   49,587    5,349
    Actual=1 (high risk)     432      876
t=0.4        0.099       0.776     0.176     0.8913    1015     293
  Confusion matrix:
                            Pred=0   Pred=1
    Actual=0 (low risk)   45,745    9,191
    Actual=1 (high risk)     293    1,015
t=0.3        0.070       0.880     0.129     0.8913    1151     157
  Confusion matrix:
                            Pred=0   Pred=1
    Actual=0 (low risk)   39,602   15,334
    Actual=1 (high risk)     157    1,151
t=0.25       0.058       0.917     0.110     0.8913    1200     108
  Confusion matrix:
                            Pred=0   Pred=1
    Actual=0 (low risk)   35,604   19,332
    Actual=1 (high risk)     108    

In [8]:
w = [2, 2, 1, 1]
voting_pipe = VotingClassifier(
    estimators=[("lr", lr_pipe), 
                ("xgb", xgb_pipe), 
                ("et", et_pipe), 
                ("rf", rf_pipe),
               ],
    voting="soft",   # average probabilities, not just votes
    n_jobs=6,
    weights=w,
)

vote_pipe_fitted, vote_metrics = evaluate_model(
    "Voting", voting_pipe, X_train, y_train, X_test, y_test, thr=0.4)


─────────────────────────────────────────────────────────────────
  MODEL: Voting
─────────────────────────────────────────────────────────────────
  Accuracy  : 0.8299
  Precision : 0.0993   (of predicted high-risk, how many truly are)
  Recall    : 0.7821   (of actual high-risk, how many we caught)
  F1-Score  : 0.1762   (harmonic mean of precision & recall)
  ROC-AUC   : 0.8918
  RC-AUC   : 0.2455

  Confusion matrix:
                          Pred=0   Pred=1
    Actual=0 (low risk)   45,656    9,280
    Actual=1 (high risk)     285    1,023

  Full classification report:
              precision    recall  f1-score   support

    Low risk       0.99      0.83      0.91     54936
   High risk       0.10      0.78      0.18      1308

    accuracy                           0.83     56244
   macro avg       0.55      0.81      0.54     56244
weighted avg       0.97      0.83      0.89     56244



In [ ]:
# Testing here because it takes sooo long to build 
# The voting classifier.
from sklearn.model_selection import cross_val_predict


lr_probs  = lr_pipe_fitted.predict_proba(X_test)[:, 1]
rf_probs  = rf_pipe_fitted.predict_proba(X_test)[:, 1]
et_probs  = et_pipe_fitted.predict_proba(X_test)[:, 1]
xgb_probs = xgb_pipe_fitted.predict_proba(X_test)[:, 1]

"""
# This gives honest PR AUC estimates without touching the test set
lr_val_probs  = cross_val_predict(lr_pipe,  X_train, y_train, cv=3, method="predict_proba")[:, 1]
rf_val_probs  = cross_val_predict(rf_pipe,  X_train, y_train, cv=3, method="predict_proba")[:, 1]
et_val_probs = cross_val_predict(et_pipe, X_train, y_train, cv=3, method="predict_proba")[:, 1]
xgb_val_probs = cross_val_predict(xgb_pipe, X_train, y_train, cv=3, method="predict_proba")[:, 1]


# I am basing the weights off of RC AOC score represented by
# average_precision_score() function as I believe it will 
# represent the quality of our model better than mere ROC AUC.

w_lr  = average_precision_score(y_train, lr_val_probs)
w_rf  = average_precision_score(y_train, rf_val_probs)
w_et  = average_precision_score(y_train, et_val_probs)
w_xgb = average_precision_score(y_train, xgb_val_probs)


"""
# To test without weights
w_lr  = 2
w_rf  = 2
w_et  = 1
w_xgb = 1


total = w_lr + w_rf + w_xgb + w_et 

# Weighted soft vote 
ensemble_probs = (
    (w_lr  * lr_probs) +
    (w_rf  * rf_probs) +
    (w_et  * et_probs) +
    (w_xgb * xgb_probs)
) / total

# Which threshold is the best.
for thr in [0.5, 0.4, 0.3, 0.25, 0.2, 0.15]:
    preds = (ensemble_probs >= thr).astype(int)
    cm    = confusion_matrix(y_test, preds)
    tn, fp, fn, tp = cm.ravel()

    prec = precision_score(y_test, preds, zero_division=0)
    rec  = recall_score(y_test, preds, zero_division=0)
    f1   = f1_score(y_test, preds, zero_division=0)
    auc  = roc_auc_score(y_test, ensemble_probs)

    print(f"{'─'*55}")
    print(f"  Threshold : {thr}")
    print(f"  AUC       : {auc:.4f}")
    print(f"  Precision : {prec:.3f}")
    print(f"  Recall    : {rec:.3f}")
    print(f"  F1        : {f1:.3f}")
    print(f"  Confusion matrix:")
    print(f"    {'':22s}  Pred=0   Pred=1")
    print(f"    Actual=0 (low risk)   {tn:>6,}   {fp:>6,}")
    print(f"    Actual=1 (high risk)  {fn:>6,}   {tp:>6,}")
    print(f"  TP={tp:,}  FN={fn:,}  FP={fp:,}  TN={tn:,}")

In [22]:
# Model 3, Random Forest 
rf_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=500,
        criterion="entropy",
        class_weight="balanced_subsample",
        random_state=42,
        n_jobs=-1,
        min_samples_leaf=5,
    )),
])

rf_pipe_fitted, rf_metrics = evaluate_model(
    "Random Forest", rf_pipe, X_train, y_train, X_test, y_test
)



─────────────────────────────────────────────────────────────────
  MODEL: Random Forest
─────────────────────────────────────────────────────────────────
  Accuracy  : 0.9628
  Precision : 0.2770   (of predicted high-risk, how many truly are)
  Recall    : 0.3723   (of actual high-risk, how many we caught)
  F1-Score  : 0.3177   (harmonic mean of precision & recall)
  ROC-AUC   : 0.8904
  RC-AUC   : 0.2294

  Confusion matrix:
                          Pred=0   Pred=1
    Actual=0 (low risk)   53,665    1,271
    Actual=1 (high risk)     821      487

  Full classification report:
              precision    recall  f1-score   support

    Low risk       0.98      0.98      0.98     54936
   High risk       0.28      0.37      0.32      1308

    accuracy                           0.96     56244
   macro avg       0.63      0.67      0.65     56244
weighted avg       0.97      0.96      0.97     56244

